# Deep Learning for Business Applications course

## TOPIC 4: Object detection problem. Mask R-CNN demo


*Notebook for MiBA (Master of Business Analytics) students. No deep math required — only intuition, pictures, and running the code.*

### 0. The big picture: what is Mask R-CNN?

In Topic 3 we classified a whole image ("this is a macaron"). But often a business needs to know **what objects are in a picture, *where* they are, and *which pixels* belong to each one** — count the tomatoes on a conveyor belt, find every car in a parking lot, outline a defect on a product. This is the job of **instance segmentation**, and **Mask R-CNN** is the classic model for it.

Everyday analogy: think of a museum guard with a marker. **Object detection** puts a rectangular frame around each painting ("here is a painting"). **Instance segmentation** goes further and traces the exact outline of each painting with a highlighter — and does it separately for two paintings that hang side by side, even if they look identical.

In this notebook:
1. We load **Mask R-CNN** already trained on **COCO** — a huge public dataset of ~80 everyday classes (person, car, dog, pizza, ...).
2. We build a few small helpers that turn the model's raw output into colored masks, boxes and class names.
3. We run the model on a handful of movie stills and look at what it finds.
4. We peek inside the output (a mask is just a picture of 0s and 1s) to demystify what the model actually returns.

We do **no training** here — this is a "use a ready-made model" demo.


### 1. Libraries


We only need **OpenCV** on top of the pre-installed libraries. OpenCV is used here purely for drawing (rectangles, text) and blending colored masks on top of the photo.


In [ ]:
!pip3 install opencv-python


Import the standard toolbox. `torchvision` ships Mask R-CNN, and `v2` provides the image transforms. We also pick a device — this container has **no GPU**, so everything runs on the CPU (a few seconds per image).


In [ ]:
import os
import cv2
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.transforms import v2
from PIL import Image

# check if GPU available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

### 2. Get the model


Here is the key idea of the whole notebook: **we do not train anything**. `maskrcnn_resnet50_fpn` is a ready-made network whose weights were learned on the COCO dataset. `weights='DEFAULT'` tells PyTorch to download and use the best available pre-trained weights (the first run downloads ~170 MB — watch your disk quota).

The model has two parts worth knowing by name:
- **ResNet50 + FPN** — the "eyes": a convolutional backbone that turns the image into feature maps at several scales.
- **Heads** — the "hands": small networks that predict a box, a class and a pixel mask for each candidate object.

| Code | Simple meaning |
|:-----|:---------------|
| `maskrcnn_resnet50_fpn` | Mask R-CNN with a ResNet50 backbone |
| `weights='DEFAULT'` | use the pre-trained COCO weights |


In [ ]:
#### 2.1. Take your 

In [ ]:

# load Mask R-CNN pre-trained on the COCO dataset
# "DEFAULT" automatically picks the best available pre-trained weights
model = torchvision.models.detection.maskrcnn_resnet50_fpn(
    weights='DEFAULT'
)



#### Where does the model file live?

Pre-trained weights are cached on disk, so the download happens **only once**. Let's look at the cache folder to see what was saved. This is a useful habit: large model files eat disk quota quickly.


In [ ]:
!ls -la ~/.cache

In [ ]:
!ls -la ~/.cache/torch/hub/checkpoints

__NOTE:__ keep in mind disk space required for models downloads!

### 3. Model inference utils


We switch the model to **evaluation mode**. Some layers (dropout, batch-norm) behave differently during training and during inference; `eval()` makes sure we get stable, reproducible predictions.


In [ ]:

# inference mode: disable training-specific behaviour (dropout, batch-norm updates)
model.eval()



The model was trained on **COCO**, so its output labels are integers (0–90). This list maps those integers to human-readable class names. `'N/A'` marks gaps because the original COCO numbering is not continuous.


In [ ]:
COCO_INSTANCE_CATEGORY_NAMES = [
    'background', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]
print(
    'total COCO classes:',
    len(COCO_INSTANCE_CATEGORY_NAMES)
)


Now the helper functions. The first one, `get_prediction`, takes an image and returns three aligned lists: masks, boxes and class names — **one entry per detected object** that is confident enough (score above `threshold`).

The model's raw output is a list of dictionaries; for each image the interesting keys are:

| Key | Simple meaning |
|:----|:---------------|
| `boxes` | rectangles around each object, `(x1, y1, x2, y2)` in pixels |
| `labels` | class number for each object (index into the COCO list) |
| `scores` | confidence of each detection, from 0 to 1 |
| `masks` | soft pixel masks (values 0–1) — how much each pixel looks like the object |


In [ ]:

def get_prediction(model, img_path, threshold):
    """Run Mask R-CNN on a single image.

    Args:
        model: Mask R-CNN model in eval mode.
        img_path: path to the input image.
        threshold: keep only detections whose confidence is above this value.

    Returns:
        masks: boolean masks, one per object, shape (N, H, W).
        boxes: list of ((x1, y1), (x2, y2)) pixel corners.
        pred_class: list of class names, one per object.
    """
    # 1) load the image and turn it into a float tensor (C, H, W) with values in [0, 1]
    img = Image.open(img_path).convert('RGB')
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
    img = transform(img)

    # 2) run the model; it takes a *list* of images and returns a *list* of dicts
    with torch.no_grad():
        pred = model([img])

    # 3) keep only confident detections (score above threshold)
    scores = pred[0]['scores'].detach().cpu().numpy()
    keep = scores > threshold

    # masks come as (N, 1, H, W) probabilities -> binarize at 0.5,
    # drop the channel dimension and keep the confident objects
    masks = (pred[0]['masks'] > .5).detach().cpu().numpy()[:, 0][keep]

    # class numbers -> human-readable names (same filter)
    pred_class = [
        COCO_INSTANCE_CATEGORY_NAMES[i]
        for i in pred[0]['labels'].detach().cpu().numpy()[keep]
    ]

    # boxes -> ((x1, y1), (x2, y2)) corners in pixels
    pred_boxes = [
        [(b[0], b[1]), (b[2], b[3])]
        for b in pred[0]['boxes'].detach().cpu().numpy()[keep]
    ]
    return masks, pred_boxes, pred_class



`random_colour_masks` paints the pixels of one object with a random color, so different objects get different highlighters. A mask is just a picture where `1` means "this pixel belongs to the object" and `0` means "background".


In [ ]:

def random_colour_masks(image):
    """Paint the pixels of a binary mask with a random color.

    Args:
        image: boolean mask of shape (H, W), where True marks the object.

    Returns:
        An RGB image (H, W, 3) with the object colored and the rest black.
    """
    colours = [
      [0, 255, 0],
      [0, 0, 255],
      [255, 0, 0],
      [0, 255, 255],
      [255, 255, 0],
      [255, 0, 255],
      [80, 70, 180],
      [250, 80, 190],
      [245, 145, 50],
      [70, 150, 250],
      [50, 190, 190]
    ]
    r = np.zeros_like(image).astype(np.uint8)
    g = np.zeros_like(image).astype(np.uint8)
    b = np.zeros_like(image).astype(np.uint8)
    # choose one color from the whole palette and paint every object pixel with it
    r[image == 1], g[image == 1], b[image == 1] = colours[random.randrange(len(colours))]
    coloured_mask = np.stack([r, g, b], axis=2)
    return coloured_mask



Finally, `instance_segmentation_api` glues everything together: it gets the predictions, blends each colored mask on top of the original photo, draws a green box and writes the class name above it.

| Code | Simple meaning |
|:-----|:---------------|
| `cv2.addWeighted(..., 0.5, ...)` | blend the color mask 50/50 with the photo |
| `cv2.rectangle(...)` | draw the bounding box |
| `cv2.putText(...)` | write the class name |
| `threshold=.75` | only show objects the model is at least 75% sure about |


In [ ]:

def instance_segmentation_api(model, img_path,
                              threshold=.75, rect_th=2,
                              text_size=1, text_th=2):
    """Run the model on one image and draw masks, boxes and labels.

    Args:
        model: Mask R-CNN model in eval mode.
        img_path: path to the input image.
        threshold: minimum confidence to show a detection.
        rect_th: line thickness of the bounding box.
        text_size: font scale of the class label.
        text_th: line thickness of the class label.
    """
    # get masks, boxes and class names for the confident detections
    masks, boxes, pred_cls = get_prediction(model, img_path, threshold)

    # OpenCV loads images as BGR, but matplotlib expects RGB
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for i in range(len(masks)):
        # blend a random-colored version of this object's mask onto the photo
        rgb_mask = random_colour_masks(masks[i])
        img = cv2.addWeighted(img, 1, rgb_mask, 0.5, 0)

        # draw the bounding box around the object
        cv2.rectangle(
            img,
            [int(x) for x in boxes[i][0]],
            [int(x) for x in boxes[i][1]],
            color=(0, 255, 0),
            thickness=rect_th
        )
        # write the class name right above the box
        cv2.putText(
            img,
            pred_cls[i],
            [int(x) for x in boxes[i][0]],
            cv2.FONT_HERSHEY_SIMPLEX,
            text_size,
            (0, 255, 0),
            thickness=text_th
        )

    plt.figure(figsize=(20, 30))
    plt.imshow(img)
    plt.xticks([])
    plt.yticks([])
    plt.show()


### 4. Model inference examples

Let's use this excellent [comedy film](https://en.wikipedia.org/wiki/Attack_of_the_Killer_Tomatoes) for our tests.


All demo images live in the read-only course data folder. We only **read** from `__DATA` — never write there.


In [ ]:

# folder with the demo images (read-only course data)
DATA_PATH = '/home/jovyan/__DATA/DLBA_F26/topic_04'



**How to read the result picture:** each colored blob is one detected object, the green rectangle is its bounding box, and the green text is the predicted class. Only objects with confidence above the threshold are shown. Two objects of the same class (e.g. two tomatoes) get **separate** masks — that is the "instance" part of instance segmentation.


In [ ]:
img_path = f'{DATA_PATH}/tomato1.jpg'
instance_segmentation_api(model, img_path)


Instead of drawing, we can also call `get_prediction` directly and work with the raw results — useful when you want to **count** objects or feed them into a business rule.


In [ ]:
masks, boxes, pred_cls = get_prediction(model, img_path, threshold=.7)


The first object the model found in the picture belongs to class:


In [ ]:
pred_cls[0]


And here is its mask. Notice it is a **2D grid of 0s and 1s** — a black-and-white picture where white marks the pixels of the object. The model does not "know" shapes in the human sense; it only outputs this grid.


In [ ]:
masks[0]


The shape tells us the mask covers the **whole image**, pixel by pixel:


In [ ]:
masks[0].shape


Here is that same mask drawn on its own, titled with the predicted class:


In [ ]:
num = 0
plt.figure(figsize=(16, 16))
plt.imshow(masks[num])
plt.title(pred_cls[num])
plt.xticks([])
plt.yticks([])
plt.show()


Let's try a second frame from the same film:


In [ ]:
img_path = f'{DATA_PATH}/tomato2.jpg'
instance_segmentation_api(model, img_path)

More samples for [another comedy](https://en.wikipedia.org/wiki/Ace_Ventura:_Pet_Detective).


A different, larger font makes the labels readable when there are many objects:


In [ ]:
img_path = f'{DATA_PATH}/ace.jpg'
instance_segmentation_api(model, img_path, text_size=2)

...and finally:


A stricter threshold (0.9) keeps only the objects the model is very sure about — compare this with the earlier pictures and notice how some less-certain detections disappear. This trade-off between **catching everything** and **avoiding false alarms** is exactly what you tune in a real business task.


In [ ]:
img_path = f'{DATA_PATH}/foods.jpg'
instance_segmentation_api(model, img_path, threshold=.9, text_size=1)


## Additional questions to think about (no code needed)

1. In your own words, what is the difference between **image classification**, **object detection** and **instance segmentation**? Give one business example where each is the right tool.
2. The model is pre-trained on COCO's ~80 classes. What happens when you feed it a product it has never seen (e.g. a specific machine part)? How would you adapt it to your own objects?
3. Look at how the picture changes when you raise or lower the `threshold`. Why might a tomato-sorting line prefer a high threshold, while a security camera might prefer a low one?
4. The mask is just a grid of 0s and 1s. How could you turn these masks into a business number — for example, the *area* a defect covers or the *number* of items on a conveyor belt?
